# Part 9: Correlation and Bivariate Analysis

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

DATA_PATH = Path("41586_2022_5154_MOESM3_ESM.xlsx")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
s1 = pd.read_excel(DATA_PATH, sheet_name="TableS1_clinical_data")
sa = s1[s1["Country"] == "South Africa"].copy()
sa["PSA"] = pd.to_numeric(sa["PSA"], errors="coerce")
sa["PSA_log"] = np.log10(sa["PSA"].replace(0, np.nan))
sa["ISUP_GG"] = pd.to_numeric(sa["ISUP_GG"].replace({"1-2": 2}), errors="coerce")
sa["Risk_class"] = sa["ISUP_GG"].apply(
    lambda x: np.nan if pd.isna(x) else ("High Risk" if x >= 3 else "Low Risk")
)
sa["chromothripsis_int"] = sa["chromothripsis_positive"].astype(int)

/Users/tombalay/miniforge3/envs/practical/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/var/folders/jp/y16t99n91ssgb6wzwplx64pc0000gn/T/ipykernel_56746/901966969.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sa["ISUP_GG"] = pd.to_numeric(sa["ISUP_GG"].replace({"1-2": 2}), errors="coerce")


In [3]:
sa_valid = sa.dropna(subset=["ISUP_GG"]).copy()
rho_psa, p_rho_psa = stats.spearmanr(
    sa_valid["PSA_log"].dropna(), sa_valid.loc[sa_valid["PSA_log"].notna(), "ISUP_GG"]
)
valid_tmb = sa_valid.dropna(subset=["TMB"])
rho_tmb, p_rho_tmb = stats.spearmanr(valid_tmb["TMB"], valid_tmb["ISUP_GG"])

def age_midpoint(age_str):
    if pd.isna(age_str):
        return np.nan
    try:
        a, b = str(age_str).split("-")
        return (int(a) + int(b)) / 2
    except Exception:
        return np.nan

sa_valid["Age_numeric"] = sa_valid["Age"].apply(age_midpoint)
valid_age = sa_valid.dropna(subset=["Age_numeric"])
rho_age, p_rho_age = stats.spearmanr(valid_age["Age_numeric"], valid_age["ISUP_GG"])
rho_snv, p_rho_snv = stats.spearmanr(sa_valid["somatic_SNVs"], sa_valid["ISUP_GG"])
ct = pd.crosstab(sa_valid["chromothripsis_int"], sa_valid["Risk_class"])
print("PSA:", rho_psa, p_rho_psa)
print("TMB:", rho_tmb, p_rho_tmb)
print("Age:", rho_age, p_rho_age)
print("SNV:", rho_snv, p_rho_snv)
display(ct)
if ct.shape == (2, 2):
    print("Chi2:", stats.chi2_contingency(ct)[:2])

PSA: 0.41918966322111045 8.55608540134283e-06
TMB: 0.2586479993078242 0.005253744436639941
Age: 0.22438411665701966 0.01639273613198369
SNV: 0.25943166934059736 0.00511331241300736


Risk_class,High Risk,Low Risk
chromothripsis_int,,
0,33,12
1,61,9


Chi2: (np.float64(2.635566933629243), np.float64(0.10449475474526085))


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
sns.boxplot(
    data=sa_valid.dropna(subset=["PSA_log"]), x="ISUP_GG", y="PSA_log", ax=axes[0, 0]
)
axes[0, 0].set_title(f"PSA vs ISUP (rho={rho_psa:.3f}, p={p_rho_psa:.4f})")
sns.boxplot(data=sa_valid.dropna(subset=["TMB"]), x="ISUP_GG", y="TMB", ax=axes[0, 1])
axes[0, 1].set_title(f"TMB vs ISUP (rho={rho_tmb:.3f}, p={p_rho_tmb:.4f})")
sns.boxplot(data=sa_valid, x="ISUP_GG", y="somatic_SNVs", ax=axes[1, 0])
axes[1, 0].set_title(f"SNVs vs ISUP (rho={rho_snv:.3f}, p={p_rho_snv:.4f})")
ct_plot = (
    sa_valid.groupby(["Risk_class", "chromothripsis_int"]).size().unstack(fill_value=0)
)
ct_plot.plot(kind="bar", stacked=True, ax=axes[1, 1], color=["#bdc3c7", "#e74c3c"])
axes[1, 1].set_title("Chromothripsis x Risk Class")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig11_bivariate_analysis.png", dpi=200)
plt.close(fig)